# Python Try...Except

> 📘 **Python Mastery** · Module 07 — Error Handling · Lesson 1/2

Perfect code still meets imperfect data: users type "abc" where a number belongs, files vanish, lists end too soon. Unhandled, any of these **crashes** your program mid-sentence. Python's `try`/`except` lets you expect trouble, absorb it, and carry on — the difference between a toy script and software people can trust.

## 🎯 Learning Objectives

By the end of this lesson you will be able to:

- Read a traceback bottom-up and locate both *what* failed and *where*.
- Wrap risky code in `try`/`except` and keep a program running past failures.
- Catch **specific** exceptions (`ValueError`, `TypeError`, `ZeroDivisionError`, `KeyError`, `IndexError`, `FileNotFoundError`) instead of everything.
- Explain why bare `except:` and broad `except Exception:` hide bugs — and avoid them.
- Use the `else` clause for success-only work and `raise` to signal your own errors.
- Build a reusable, robust number-input validator.

## 1. Why Programs Crash — Reading a Traceback

When an error occurs and nobody handles it, Python prints a **traceback** and terminates the program. Beginners fear tracebacks; professionals *read* them, because they are a precise accident report:

```text
Traceback (most recent call last):
  File "grades.py", line 12, in <module>
    average = total / count
ZeroDivisionError: division by zero
```

Read it **bottom-up**:

1. **Last line** — *what* went wrong: the exception type plus its message.
2. **Middle lines** — *where*: the exact file, line, and source text (`grades.py`, line 12).
3. **Top line** — the call chain that led there (short here; long in real projects).

One unhandled error kills the whole run — every later line simply never executes. That brittleness is what `try`/`except` fixes.

## 2. `try` / `except`: Your Safety Net

Put the risky statements inside `try:`. If any of them raises the matching error, Python jumps straight to `except:` and runs *your* recovery plan instead of dying. If nothing fails, the `except` block is skipped entirely.

**Syntax:**

```python
try:
    risky_operation()              # attempt this...
except SomeError:
    recovery_plan()                # ...only if it raised SomeError
```

**Example:** one corrupted sensor reading would normally abort the whole loop.

In [1]:
print("Program starts.")

readings = [10, 0, 4]              # a sensor glitched and reported 0

for r in readings:
    try:
        share = 100 / r
        print(f"100 / {r} = {share}")
    except ZeroDivisionError:
        print(f"100 / {r} -> mathematically impossible, skipping")

print("Program ends normally - one bad reading did not sink the ship.")

Program starts.
100 / 10 = 10.0
100 / 0 -> mathematically impossible, skipping
100 / 4 = 25.0
Program ends normally - one bad reading did not sink the ship.


## 3. Catch *Specific* Exceptions

`except` needs to know **which** error it can fix — each exception type means a different disease requiring different medicine. These are the built-ins you will meet constantly:

| Exception | Raised when... | Typical trigger |
|-----------|----------------|-----------------|
| `ValueError` | right type, unacceptable value | `int("abc")` |
| `TypeError` | wrong type altogether | `"hello" - 1` |
| `ZeroDivisionError` | dividing by zero | `100 / 0` |
| `KeyError` | dictionary key does not exist | `profile["age"]` |
| `IndexError` | sequence index out of range | `marks[99]` |
| `FileNotFoundError` | opening a missing file | `open("ghost.txt")` |
| `NameError` | using an undefined variable | `print(totl)` (typo) |
| `AttributeError` | method/attribute not on that object | `"hi".push()` |
| `ModuleNotFoundError` | import target does not exist | `import pandass` |

**Syntax:**

```python
try:
    number = int(text)
except ValueError:
    ...   # exactly the failure int() can produce
```

**Example:** four different accidents, each caught by name.

In [2]:
experiments = [
    ("int('abc')",     lambda: int("abc")),                    # ValueError
    ("'hello' - 1",    lambda: "hello" - 1),                   # TypeError
    ("profile['age']", lambda: {"name": "Sarah"}["age"]),      # KeyError
    ("marks[99]",      lambda: [90, 85, 70][99]),              # IndexError
]

for label, action in experiments:
    try:
        action()
    except (ValueError, TypeError, KeyError, IndexError) as e:
        print(f"{label:<16} -> {type(e).__name__}: {e}")

int('abc')       -> ValueError: invalid literal for int() with base 10: 'abc'
'hello' - 1      -> TypeError: unsupported operand type(s) for -: 'str' and 'int'
profile['age']   -> KeyError: 'age'
marks[99]        -> IndexError: list index out of range


## 4. Multiple `except` Blocks

When one `try` can fail in several distinct ways, give each failure its own handler — Python checks them **top to bottom** and runs the first match. Order therefore matters: put specific exceptions first; a general one placed first would swallow everything before the specific handlers ever see it.

**Syntax:**

```python
try:
    number = int(text)         # may raise ValueError
    result = 100 / number      # may raise ZeroDivisionError
except ValueError:
    ...
except ZeroDivisionError:
    ...
```

**Example:** three user submissions, three different outcomes.

In [3]:
submissions = ["12", "0", "seven"]

for raw in submissions:
    try:
        number = int(raw)                  # ValueError for junk text
        print(f"100 / {number} = {100 / number:.1f}")   # ZeroDivisionError for 0
    except ValueError:
        print(f"'{raw}' is not a number at all")
    except ZeroDivisionError:
        print("'0' is a number, but 100 / 0 is undefined")

100 / 12 = 8.3
'0' is a number, but 100 / 0 is undefined
'seven' is not a number at all


## 5. `except ... as e`: Capturing the Details

The exception object carries the error message and much more. Bind it with `as e` (any short name works) to log it, show it to the user, or make decisions based on it. `type(e).__name__` gives the class name as a string — invaluable for logging.

**Syntax:**

```python
try:
    ...
except ValueError as e:
    print(type(e).__name__, "-", e)
```

**Example:**

In [4]:
try:
    price = int("ninety-nine dollars")
except ValueError as e:
    print("technical type :", type(e).__name__)
    print("raw message    :", e)
    print("human version  : please enter digits only, e.g. 99")

technical type : ValueError
raw message    : invalid literal for int() with base 10: 'ninety-nine dollars'
human version  : please enter digits only, e.g. 99


## 6. The Danger of Bare `except:` and Broad `except Exception:`

Two shortcuts look convenient and are quietly destructive:

- `except:` — catches **everything**, including `Ctrl+C` (`KeyboardInterrupt`) and typos.
- `except Exception:` — nearly everything else.

Why dangerous: they cannot tell a *handled problem* from an *unnoticed bug*. Misspell a variable, corrupt your data structure, break your own logic — the broad net swallows the evidence and the program sails on confidently in the wrong state. During development especially, loud crashes are a **feature**: they point at the wound while it is fresh.

**Syntax (avoid the first two forms):**

```python
except:                     # NEVER - hides even Ctrl+C
except Exception: pass      # rarely - hides real bugs
except ValueError as e: ... # YES - precise and honest
```

**Example:** the same typo, first hidden forever, then surfaced.

In [5]:
total = 150

# BAD: a typo becomes an invisible bug.
try:
    totel = totel + 10            # NameError: misspelled variable...
except Exception:
    pass                          # ...vanished without a trace

print("after BAD block , total =", total, "- zero evidence of the bug")

# GOOD: catch precisely what you expect; let surprises stay loud.
try:
    total = total + 10
except NameError as e:
    print("GOOD block saw  :", type(e).__name__, "-", e)

print("after GOOD block, total =", total)

after BAD block , total = 150 - zero evidence of the bug
after GOOD block, total = 160


## 7. `else`: Runs Only When Nothing Failed

`else` completes the sentence: *"if the `try` block raised nothing, do this."* It solves a subtle design problem — work that **uses** the try-block's result should not sit *inside* the `try`, or its own bugs would be caught by the same handlers and misdiagnosed as input problems. Put result-handling in `else`: it runs only on success, and its errors stay loud.

**Syntax:**

```python
try:
    value = int(text)
except ValueError:
    print("rejected")
else:
    print("valid:", value * 2)     # safe to USE value here
```

**Example:**

In [6]:
submissions = ["42", "not-a-number", "7"]

for raw in submissions:
    try:
        number = int(raw)
    except ValueError:
        print(f"'{raw}' rejected - not an integer")
    else:
        # Runs ONLY on success, so `number` genuinely exists here.
        print(f"'{raw}' accepted -> doubled is {number * 2}")

'42' accepted -> doubled is 84
'not-a-number' rejected - not an integer
'7' accepted -> doubled is 14


## 8. `raise`: Throwing Errors on Purpose

You are not limited to Python's mistakes — your own functions should refuse nonsense **loudly and immediately** rather than returning garbage that explodes later. `raise ExceptionType("message")` fires an exception just like a built-in one, and callers catch it with the same machinery. Failing fast with a clear message is a kindness to whoever debugs this next — usually future you.

**Syntax:**

```python
def withdraw(amount):
    if amount <= 0:
        raise ValueError(f"amount must be positive, got {amount}")
    ...
```

**Example:**

In [7]:
def set_exam_score(score):
    """Record an exam score - refusing anything nonsensical."""
    if not isinstance(score, int):
        raise TypeError(f"score must be an int, got {type(score).__name__}")
    if not 0 <= score <= 100:
        raise ValueError(f"score {score} is outside the range 0-100")
    return score

for candidate in [92, 142, "ninety-two"]:
    try:
        set_exam_score(candidate)
        print(candidate, "-> accepted")
    except (TypeError, ValueError) as e:
        print(candidate, "-> rejected:", type(e).__name__, "-", e)

92 -> accepted
142 -> rejected: ValueError - score 142 is outside the range 0-100
ninety-two -> rejected: TypeError - score must be an int, got str


## 9. 🔍 Under the Hood: The Exception Hierarchy

Every exception is a class, and they form a family tree. `ValueError` is a *kind of* `Exception` — which is exactly why `except Exception` catches it. Knowing a few key relationships lets you catch a whole branch with one name (e.g., `except LookupError` covers both `IndexError` and `KeyError`).

```text
BaseException
 ├── KeyboardInterrupt        # Ctrl+C  (almost never catch!)
 ├── SystemExit              # sys.exit()
 └── Exception               # root of all ordinary errors
      ├── ValueError         # right type, bad value
      ├── TypeError          # wrong type entirely
      ├── ArithmeticError
      │    └── ZeroDivisionError
      ├── LookupError
      │    ├── IndexError
      │    └── KeyError
      └── OSError
           └── FileNotFoundError
```

> 🔍 **Under the Hood:** `raise SomeError("...")` creates an *instance* of an exception class and begins unwinding the call stack. At each frame Python tests every live handler with `issubclass(type_of_error, caught_type)` — which is precisely why `except Exception` catches `ValueError`, and why handler order decides who answers first. The travelling object carries a **traceback**: a record of every frame it passed through — the very report you learned to read bottom-up in Section 1.

**Example:** verify the family ties yourself.

In [8]:
print("ValueError        child of Exception  :", issubclass(ValueError, Exception))
print("ZeroDivisionError child of Arithmetic :", issubclass(ZeroDivisionError, ArithmeticError))
print("KeyError          child of LookupError:", issubclass(KeyError, LookupError))
print("FileNotFoundError child of OSError    :", issubclass(FileNotFoundError, OSError))
print("KeyboardInterrupt child of Exception  :", issubclass(KeyboardInterrupt, Exception),
      "<- deliberately NOT: Ctrl+C bypasses normal handlers")

ValueError        child of Exception  : True
ZeroDivisionError child of Arithmetic : True
KeyError          child of LookupError: True
FileNotFoundError child of OSError    : True
KeyboardInterrupt child of Exception  : False <- deliberately NOT: Ctrl+C bypasses normal handlers


## 10. Practical Pattern: A Robust Number Validator

Remember Lesson "User Input"? `input()` returns strings, users type chaos, and `int(raw)` explodes on the first creative answer. Now you have every piece for the professional fix: convert inside `try`, reject with a helpful `raise`, decide in `except`/`else`. Since notebooks cannot pause for typing, we simulate the user's attempts with a list.

**Syntax:**

```python
while True:
    try:
        value = int(input("Age: "))
    except ValueError:
        print("digits only, please")
    else:
        break          # valid at last
```

**Example:**

In [9]:
# Simulated input: what the user types across their attempts.
attempts = ["abc", "-5", "3.5", "42"]

def parse_positive_int(text):
    """Return a positive whole number, or raise ValueError stating exactly why."""
    if "." in text:
        raise ValueError(f"'{text}' looks like a decimal - whole numbers only")
    value = int(text)                 # raises ValueError for junk like 'abc'
    if value <= 0:
        raise ValueError(f"{value} is not positive")
    return value

for text in attempts:
    try:
        answer = parse_positive_int(text)
    except ValueError as e:
        print(f"x '{text}' rejected -> {e}")
    else:
        print(f"OK '{text}' accepted -> age recorded as {answer}")
        break

x 'abc' rejected -> invalid literal for int() with base 10: 'abc'
x '-5' rejected -> -5 is not positive
x '3.5' rejected -> '3.5' looks like a decimal - whole numbers only
OK '42' accepted -> age recorded as 42


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---------|---------|-----|
| `except:` or `except Exception: pass` | Real bugs (typos, bad logic) vanish silently | Catch named, specific exceptions |
| Guarding 30 lines in one `try` | Any failure, anywhere, triggers the same vague handler | Wrap only the genuinely risky lines |
| Listing `except Exception` before specific ones | It matches first; specific handlers never run | Most specific handler on top |
| `raise ValueError` without a message | Future debugging finds an empty clue | Always raise with a descriptive string |
| Using the result of a failed `try` afterwards | Variable may not exist / hold garbage | Consume results in `else` |
| Expecting `except` to also do cleanup | Handler runs *instead of* cleanup paths | That is `finally`'s job — next lesson |

## 💡 Best Practices & Pro Tips

- Catch the narrowest exception that names the failure you can actually fix; let genuine bugs stay loud during development.
- Bind with `as e` and include `type(e).__name__` + message whenever you report or log a failure — context is cheap now and priceless later.
- Keep `try` blocks minimal and push success-path work into `else`; readers instantly see what is risky and what is routine.
- Validate arguments with early `raise`s carrying concrete values (`f"got {amount}"`) — fail fast, fail clearly. Custom exception classes arrive naturally once you learn classes (Module 08).
- **AI-engineering relevance:** production ML code is mostly error handling wrapped around science — skipping one malformed row out of millions in a training loop, falling back to a previous checkpoint when a download fails, retrying rate-limited API calls, validating that a model file loads before serving traffic. Every one of those is `try`/`except` with a *specific* exception and a *planned* response.

## 📌 Summary

| Construct | What it does | Example |
|-----------|--------------|---------|
| `try:` | Attempt risky code | `try: x = int(s)` |
| `except E:` | Run recovery for that error | `except ValueError:` |
| `except E as e:` | Recovery with the error object | `print(e)` |
| Multiple `except` blocks | One handler per failure mode (specific first!) | `except KeyError:` ... |
| `else:` | Runs only when `try` fully succeeded | `else: use(value)` |
| `raise E("msg")` | Signal your own error, loudly | `raise ValueError(f"bad {x}")` |
| Hierarchy | `ValueError` ⊂ `Exception`; catch branches by parent | `except LookupError:` |

Key takeaways:

- Tracebacks are reports, not insults — read the last line first.
- Specific exceptions are documentation: they say what you expect and how you cope.
- Broad handlers convert visible bugs into invisible corruption; precision is safety.
- `raise` makes your functions honest about bad input, and `else` keeps success logic out of the blast radius.

## 🔗 Next Lesson

Handling the error is half the story — releasing resources *whatever* happens is the other half. Finish the toolkit with `finally` and friends: [`../02_Finally/notes.ipynb`](../02_Finally/notes.ipynb).